# Try the right whale upcall detector

Run this notebook in the repository's environment, after the README's Setup (which also
downloads the model into `model/`):

```bash
uv run jupyter lab notebooks/demo.ipynb
```

It downloads 20 minutes of a NOAA recording (about 40 MB), runs the detector, and lets you look
at and listen to each detection.

## 1. Load the model

The repository's `model/` folder holds the network's weights (`weights.pt`) and its settings
(`config.json`). The settings include the operating point τ and the SHA-256 of the weights,
which `load_model` checks.

In [ ]:
from pathlib import Path

from narw_detector.weights import load_model, pick_device

MODEL_DIR = Path("../model")  # the repository's model/ folder
if not (MODEL_DIR / "weights.pt").exists():
    raise FileNotFoundError("no model in model/: download it as the README's Setup describes")

device = pick_device()
net, spec, tau = load_model(MODEL_DIR, device)
print(f"{sum(p.numel() for p in net.parameters()):,} parameters on {device}, tau = {tau:.4f}")

## 2. Get a recording

The recording is from the NOAA Northeast Fisheries Science Center (NEFSC) Passive Acoustics
Branch, archived by NOAA's National Centers for Environmental Information (NCEI) in a public
Google Cloud bucket. It is a SoundTrap file from deployment NEFSC_MA-RI_202110_NS01, recorded on
2 February 2022. That deployment was held out from training: it is one of the two NOAA
deployments in the README's results.

The file is 4 hours long (308 MB) at 64 kHz. To keep the download small, we fetch only its first
40 MB, decode the first 20 minutes and resample them to 2 kHz, the rate the model reads.
`narw-detect` resamples whole files by itself; here we cut the file first.

To cite the recording, cite NOAA's dataset: NOAA NEFSC, *NEFSC Southern New England bottom
mounted PAM audio data*, NOAA National Centers for Environmental Information,
[doi:10.25921/znjw-at57](https://doi.org/10.25921/znjw-at57).

In [ ]:
import urllib.request

import soundfile as sf

from narw_detector.audio import SAMPLE_RATE, resample

URL = ("https://storage.googleapis.com/noaa-passive-bioacoustic/nefsc/audio/nefsc_sne/"
       "nefsc_ma-ri_202110_ns01/audio/6124.220202215803.flac")
MINUTES = 20
DATA = Path("demo_data")  # the download and the outputs; git ignores this folder

head = DATA / "6124.220202215803_head.flac"
if not head.exists():
    DATA.mkdir(exist_ok=True)
    request = urllib.request.Request(URL, headers={"Range": "bytes=0-39999999"})
    with urllib.request.urlopen(request) as response:
        head.write_bytes(response.read())

with sf.SoundFile(head) as f:
    sr = f.samplerate
    x_original = f.read(frames=MINUTES * 60 * sr, dtype="float32")
assert len(x_original) == MINUTES * 60 * sr, "the download ended before 20 minutes"
x = resample(x_original, sr)

(DATA / "recordings").mkdir(exist_ok=True)
wav = DATA / "recordings" / f"6124.220202215803_first{MINUTES}min.wav"
sf.write(wav, x, SAMPLE_RATE)
print(f"{MINUTES} min at {sr:,} Hz, resampled to {SAMPLE_RATE:,} Hz -> {wav}")

## 3. Run the detector

This runs `narw-detect`, the command you would run on your own recordings. It writes one row per
detected call to `detections.csv` (start and end in seconds, and the detection's confidence),
and one Raven Pro selection table per recording.

In [ ]:
import pandas as pd

from narw_detector import detect

# The same as the command: narw-detect RECORDINGS --model MODEL_DIR --out OUT_DIR
detect.main([str(DATA / "recordings"), "--model", str(MODEL_DIR),
             "--out", str(DATA / "detections")])
pd.read_csv(DATA / "detections" / "detections.csv").head(10)

## 4. Every event, and the threshold

The detector first finds events: runs of frames whose probability is above 0.4. An event's
confidence is its highest frame probability, and an event becomes a detection when its confidence
reaches τ. The released τ, 0.5926, was chosen on our internal validation set.

Move the slider to see how the detections change with the threshold. The spectrogram shows the
whole 20 minutes at 25–500 Hz with one brightness setting, so a quiet minute looks quiet next to a
loud one. NOAA's annotations of this recording are not distributed with this repository, so the
plots show what the model found, not which detections are right. Look at and listen to them in the
next section.

In [ ]:
import numpy as np
import torch

from narw_detector.audio import window
from narw_detector.events import frame_times
from narw_detector.scan import WINDOW_S, detections, scan_events, window_probs, window_starts

events = scan_events(net, x, spec, device)  # every merged event above the 0.4 floor
found = detections(events, tau)
print(f"{len(events)} events, {len(found)} detections at tau = {tau:.4f}")

# Frame probabilities over the whole stretch: scan windows overlap, so take the highest value.
n_window = int(WINDOW_S * SAMPLE_RATE)
times, probs = [], []
for start in window_starts(len(x)):
    audio = torch.from_numpy(window(x, int(start * SAMPLE_RATE), n_window))[None].to(device)
    p = window_probs(net, audio, spec)[0]
    lo, hi = frame_times(spec.n_frames(n_window), len(p))
    times.append(start + (lo + hi) / 2)
    probs.append(p)
trace = (pd.DataFrame({"t": np.concatenate(times), "p": np.concatenate(probs)})
         .assign(t=lambda d: (d.t / 0.125).round() * 0.125)
         .groupby("t").p.max())

In [ ]:
import functools

import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import FloatSlider, IntSlider, interact
from scipy.signal import spectrogram

sns.set_theme()
AMBER, GREY, SLATE = "#FBBF24", "0.55", "#94A3B8"
SOURCE = f"released threshold {tau:.4f}, chosen on our internal validation set"
DURATION = len(x) / SAMPLE_RATE

# The display spectrogram is finer than the model's input: 256 ms windows every 32 ms, zero-padded
# to about 2 Hz per row (the model reads 8 Hz rows), so an upcall's shape is easy to see.
N_FFT, N_OVERLAP, N_PAD = 512, 448, 1024
MAX_COLUMNS = 1400  # about a figure's width in pixels


@functools.lru_cache(maxsize=64)
def display_db(t0, t1, f0, f1):
    """The display spectrogram of x from t0 to t1 s and f0 to f1 Hz, in dB, and its extent.

    Rows run from low to high frequency. A view wider than MAX_COLUMNS frames keeps the loudest
    frame of each group, so a one-second call stays visible in a 20-minute view.
    """
    a, b = int(t0 * SAMPLE_RATE), int(t1 * SAMPLE_RATE)
    freqs, times, sxx = spectrogram(x[a:b], fs=SAMPLE_RATE, window="hann", nperseg=N_FFT,
                                    noverlap=N_OVERLAP, nfft=N_PAD)
    rows = (freqs >= f0) & (freqs <= f1)
    db = 10 * np.log10(sxx[rows] + 1e-20)
    k = -(-db.shape[1] // MAX_COLUMNS)  # frames per column
    n = db.shape[1] // k * k
    db = db[:, :n].reshape(len(db), -1, k).max(axis=2)
    centres = t0 + times[:n].reshape(-1, k).mean(axis=1)
    half = k * (N_FFT - N_OVERLAP) / SAMPLE_RATE / 2
    return db, (centres[0] - half, centres[-1] + half, freqs[rows][0], freqs[rows][-1])


def show_spectrogram(ax, t0, t1, f0, f1, bright=0.0):
    """Draw the display spectrogram; frequency increases upward.

    The levels come from the picture itself: black at its median (the noise floor) and white at
    its 99.5th percentile. bright, in dB, moves both.
    """
    db, extent = display_db(t0, t1, f0, f1)
    black, white = np.percentile(db, [50, 99.5])
    ax.imshow(db, origin="lower", aspect="auto", cmap="magma", extent=extent,
              vmin=black - bright, vmax=white - bright)
    ax.set(xlim=extent[:2], ylim=(f0, f1), ylabel="frequency (Hz)")
    ax.grid(False)


def mark(ax, s0, s1, brackets=True):
    """A detection: a bar along the top and, with brackets, its start and end over 50-250 Hz (the
    band its Raven selection gets). No fill, so the call underneath stays visible."""
    ax.hlines(0.97 * ax.get_ylim()[1], s0, s1, color=AMBER, lw=5)
    if brackets:
        tick = (s1 - s0) / 4
        ax.vlines([s0, s1], 50, 250, color=AMBER, lw=1.5)
        for y in (50, 250):
            ax.hlines(y, s0, s0 + tick, color=AMBER, lw=1.5)
            ax.hlines(y, s1 - tick, s1, color=AMBER, lw=1.5)


def plot_probability(ax, threshold, t0, t1):
    near = trace[(trace.index >= t0) & (trace.index <= t1)]
    ax.plot(near.index, near.values, color="0.25", lw=0.8)
    ax.axhline(threshold, color=AMBER, ls="--", lw=1.2, label=f"threshold {threshold:.4f}")
    ax.axhline(0.4, color=GREY, ls=":", lw=1.2, label="event floor 0.4")
    ax.set(xlim=(t0, t1), ylim=(0, 1), ylabel="probability")
    ax.legend(loc="upper right", fontsize=8)


@interact(threshold=FloatSlider(value=round(tau, 4), min=0.4, max=0.95, step=0.0001,
                                readout_format=".4f", description="threshold"))
def overview(threshold):
    kept = detections(events, threshold)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(14, 5.5), sharex=True,
                                      gridspec_kw={"height_ratios": [2, 1]})
    show_spectrogram(top, 0, DURATION, 25, 500)
    for s0, s1, _ in kept:
        mark(top, s0, s1, brackets=False)
    plot_probability(bottom, threshold, *top.get_xlim())
    centres = [(s0 + s1) / 2 for s0, s1, _ in events]
    confidence = np.array([c for _, _, c in events])
    bottom.scatter(centres, confidence, s=14, zorder=3,
                   c=np.where(confidence >= threshold, AMBER, GREY))
    bottom.set_xlabel("seconds from the start of the recording")
    top.set_title(f"{len(kept)} detections (amber) at threshold {threshold:.4f} ({SOURCE})")
    plt.show()

## 5. Look at and listen to each detection

Each detection is shown twice. Zoomed out, a minute around it over 25–1000 Hz, where a passing
ship or a singing humpback shows. Zoomed in, 8 seconds over 25–500 Hz, where you can see the call's
shape. Each picture sets its own brightness from its own noise floor, so a faint call is visible
whether the minute around it is quiet or loud.

Upcalls sit around 100–200 Hz, at the bottom of human hearing. The players play the zoomed-in
8 seconds; the second plays them twice as fast, which also makes the call an octave higher and
easier to hear.

In [ ]:
from IPython.display import Audio, display
from scipy.signal import resample_poly

assert found, "no detections to browse"


def around(s0, s1, span):
    """span seconds centred on s0-s1 (at least 2 s more than it), kept inside the recording."""
    span = max(span, s1 - s0 + 2)
    start = min(max((s0 + s1 - span) / 2, 0.0), DURATION - span)
    return start, start + span


@interact(n=IntSlider(value=1, min=1, max=len(found), description="detection"),
          bright=FloatSlider(value=0, min=-10, max=10, step=1, description="brightness"))
def browse(n, bright):
    s0, s1, confidence = found[n - 1]
    w0, w1 = around(s0, s1, 60)  # zoomed out
    z0, z1 = around(s0, s1, 8)   # zoomed in
    fig, (wide, prob, close) = plt.subplots(3, 1, figsize=(12, 8.5),
                                            gridspec_kw={"height_ratios": [2.2, 0.8, 2.6]})
    show_spectrogram(wide, w0, w1, 25, 1000, bright)
    show_spectrogram(close, z0, z1, 25, 500, bright)
    for d0, d1, _ in found:
        mark(wide, d0, d1)
        mark(close, d0, d1)
    wide.vlines([z0, z1], 25, 1000, color=SLATE, ls="--", lw=1.2)
    plot_probability(prob, tau, *wide.get_xlim())
    wide.set_title(f"zoomed out: {w1 - w0:.0f} s, 25–1000 Hz (dashed: the zoomed-in part)",
                   loc="left", fontsize=10)
    close.set_title(f"zoomed in: {z1 - z0:.0f} s, 25–500 Hz", loc="left", fontsize=10)
    close.set_xlabel("seconds from the start of the recording")
    fig.suptitle(f"detection {n} of {len(found)}: {s0:.1f}–{s1:.1f} s, confidence "
                 f"{confidence:.2f}; amber marks detections\n({SOURCE})")
    fig.tight_layout()
    plt.show()
    clip = x[int(z0 * SAMPLE_RATE):int(z1 * SAMPLE_RATE)]
    print("Real speed (resampled to 4 kHz, because browsers do not play 2 kHz audio):")
    display(Audio(resample_poly(clip, 2, 1), rate=2 * SAMPLE_RATE))
    print("Twice as fast, so an octave higher (the 2 kHz samples played at 4 kHz):")
    display(Audio(clip, rate=2 * SAMPLE_RATE))

## Your own recordings

- Detect: `uv run narw-detect your_recordings/ --model model/ --out detections/`. Any sample
  rate works, and each file gets a Raven Pro selection table.
- Choose a threshold for your recordings: `uv run narw-threshold` (see the README).
- Train on your own annotated recordings: `uv run narw-train` (see the README and
  `docs/training.md`).

On recorders, sites and noise conditions unlike the training data, the model may miss more calls
or raise more false detections. Check it on a few annotated days of your own before relying on it.